# LeetCode 3046 — Split the Array bằng Python

Link đề: https://leetcode.com/problems/split-the-array/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def isPossibleToSplit(self, nums: List[int]) -> bool:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia mảng độ dài **chẵn** thành hai nửa bằng nhau về số phần tử, sao cho **mỗi nửa không có phần tử trùng nhau** (các phần tử trong cùng một nửa đều **khác nhau đôi một**).
2. Nắm các công cụ đếm: `list.count`, `set`, `dict`, `collections.Counter`, và `sorted` để các phần tử bằng nhau đứng cạnh nhau.
3. Đi từ cách **thử mọi cách chia** (brute force, chỉ chạy được khi mảng bé) → **đếm bằng `count`** → **`Counter`** → **sắp xếp rồi so hàng xóm** → **mô phỏng việc xếp vào hai tập**.
4. Hiểu mấu chốt **"mỗi giá trị xuất hiện tối đa 2 lần"** và **vì sao điều kiện đó vừa cần vừa đủ** (nguyên lý chuồng bồ câu + cách xây dựng cách chia).
5. Tránh các lỗi kinh điển: chỉ kiểm tra `len(set(nums)) >= n/2`, nhầm "tối đa 2" thành "đúng 2", chia nửa theo vị trí (nửa đầu / nửa sau) thay vì chia tự do.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có **độ dài chẵn**. Bạn cần chia `nums` thành hai mảng `nums1` và `nums2` sao cho:

- `nums1.length == nums2.length == nums.length / 2`;
- `nums1` chứa **các phần tử đôi một khác nhau**;
- `nums2` chứa **các phần tử đôi một khác nhau**.

(Mỗi phần tử của `nums` thuộc đúng một trong hai mảng; thứ tự trong mảng con không quan trọng.)

Trả về `true` nếu chia được, ngược lại `false`.

**Ví dụ 1**

- Input: `nums = [1, 1, 2, 2, 3, 4]`
- Output: `true`
- Giải thích: một cách chia là `nums1 = [1, 2, 3]` và `nums2 = [1, 2, 4]`. Mỗi mảng có 3 phần tử khác nhau.

**Ví dụ 2**

- Input: `nums = [1, 1, 1, 1]`
- Output: `false`
- Giải thích: mỗi mảng chỉ có 2 phần tử, nhưng cả bốn phần tử đều là `1` nên mảng nào cũng có hai số `1` giống nhau. Không có cách chia hợp lệ.

**Ràng buộc:** `1 <= nums.length <= 100`; `nums.length % 2 == 0`; `1 <= nums[i] <= 100`.

In [1]:
from typing import List
from collections import Counter
from itertools import combinations
import random

examples = [
    ([1, 1, 2, 2, 3, 4], True),
    ([1, 1, 1, 1], False),
    ([1, 2], True),
]
for nums, expected in examples:
    print("nums =", nums, "-> kỳ vọng", expected)

nums = [1, 1, 2, 2, 3, 4] -> kỳ vọng True
nums = [1, 1, 1, 1] -> kỳ vọng False
nums = [1, 2] -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. "Đôi một khác nhau" nghĩa là gì? Dùng `set` để kiểm tra

Một mảng có các phần tử **đôi một khác nhau** nếu không có hai vị trí nào chứa cùng một giá trị. Một cách kiểm tra ngắn gọn: `len(set(a)) == len(a)`. `set` là tập hợp: nó **tự loại các phần tử trùng**, nên nếu có trùng thì `len(set(a))` nhỏ hơn `len(a)`.

In [2]:
a = [1, 2, 3]
b = [1, 2, 2]
print(set(b))                          # {1, 2}: số 2 chỉ còn một lần
print(len(set(a)) == len(a))           # True  : không trùng
print(len(set(b)) == len(b))           # False : có trùng

{1, 2}
True
False


### 3.2. Đếm số lần xuất hiện: `list.count` và `Counter`

- `a.count(x)` trả về số lần `x` xuất hiện trong `a` (duyệt cả danh sách, nên tốn `O(n)`).
- `Counter(a)` đếm **tất cả** giá trị trong **một lượt duyệt** và cho một dict dạng `{giá trị: số lần}`. `Counter.values()` là các số lần.

In [3]:
a = [1, 1, 2, 2, 3, 4]
print(a.count(1), a.count(3), a.count(9))      # 2 1 0
cnt = Counter(a)
print(cnt)                                      # Counter({1: 2, 2: 2, 3: 1, 4: 1})
print(list(cnt.values()), max(cnt.values()))   # [2, 2, 1, 1] 2

2 1 0
Counter({1: 2, 2: 2, 3: 1, 4: 1})
[2, 2, 1, 1] 2


### 3.3. `sorted` đưa các phần tử bằng nhau lại gần nhau

Sau khi sắp xếp, mọi giá trị bằng nhau **nằm liền kề** nhau. Muốn biết có giá trị xuất hiện **từ 3 lần trở lên** hay không, chỉ cần xem có chỗ nào `s[i] == s[i + 2]` hay không (ba số liên tiếp bằng nhau).

In [4]:
a = [3, 1, 2, 1, 2, 1]
s = sorted(a)
print(s)                                                     # [1, 1, 1, 2, 2, 3]
print([i for i in range(len(s) - 2) if s[i] == s[i + 2]])   # [0] : ba số 1 liền nhau bắt đầu ở 0

[1, 1, 1, 2, 2, 3]
[0]


### 3.4. `any` và `all`

- `all(iterable)` là `True` nếu **mọi** phần tử đều đúng; `all([])` là `True`.
- `any(iterable)` là `True` nếu **có ít nhất một** phần tử đúng; `any([])` là `False`.

Ta sẽ viết "mọi giá trị đều xuất hiện không quá 2 lần" bằng `all(v <= 2 for v in cnt.values())`.

In [5]:
print(all(v <= 2 for v in [2, 1, 2]), all(v <= 2 for v in [2, 3]))   # True False
print(any(v > 2 for v in [2, 1, 2]), any(v > 2 for v in [2, 3]))     # False True
print(all([]), any([]))                                              # True False

True False
False True
True False


### 3.5. `itertools.combinations` để liệt kê các cách chọn

`combinations(range(n), k)` cho mọi cách chọn `k` chỉ số từ `n` chỉ số (không quan tâm thứ tự). Với `n = 4`, `k = 2` có `C(4, 2) = 6` cách. Ta dùng nó để chọn **những chỉ số thuộc `nums1`**; các chỉ số còn lại thuộc `nums2`.

In [6]:
print(list(combinations(range(4), 2)))
# [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tự chia `[1, 1, 2, 2, 3, 4]` thành hai nhóm 3 phần tử, mỗi nhóm không trùng:

- Hai số `1` không được vào cùng một nhóm → mỗi nhóm nhận một số `1`.
- Hai số `2` cũng vậy → mỗi nhóm nhận một số `2`.
- Còn `3` và `4`: mỗi số xuất hiện một lần, đặt tùy ý, miễn sao hai nhóm cùng cỡ: `3` vào nhóm 1, `4` vào nhóm 2.

Kết quả `[1, 2, 3]` và `[1, 2, 4]`.

Còn `[1, 1, 1, 1]`: bốn số `1` mà chỉ có **hai nhóm**. Số `1` thứ ba buộc phải rơi vào một nhóm đã có số `1` → trùng. Đó là **nguyên lý chuồng bồ câu**: có `2` chuồng (hai nhóm) mà `3` bồ câu (ba số `1`) thì có chuồng chứa `2` con.

**Rút ra:** nếu một giá trị xuất hiện **từ 3 lần trở lên** thì chắc chắn **không chia được**. Câu hỏi còn lại: nếu mọi giá trị xuất hiện **tối đa 2 lần**, có luôn chia được không? (§6 sẽ chứng minh là có.)

In [7]:
nums = [1, 1, 2, 2, 3, 4]
cnt = Counter(nums)
print("số lần xuất hiện:", dict(cnt))
print("giá trị xuất hiện 2 lần :", [v for v, c in cnt.items() if c == 2])
print("giá trị xuất hiện 1 lần :", [v for v, c in cnt.items() if c == 1])
print("giá trị xuất hiện >= 3  :", [v for v, c in cnt.items() if c >= 3])

số lần xuất hiện: {1: 2, 2: 2, 3: 1, 4: 1}
giá trị xuất hiện 2 lần : [1, 2]
giá trị xuất hiện 1 lần : [3, 4]
giá trị xuất hiện >= 3  : []


## 5. Cách 1 — Brute Force / baseline: thử mọi cách chia

**Ý tưởng.** Thử **mọi cách** chọn `n/2` chỉ số cho `nums1`; phần còn lại là `nums2`. Với mỗi cách, kiểm tra cả hai nhóm đều không trùng bằng `set`. Nếu có một cách đúng thì `True`.

Cách này **bám sát định nghĩa** của đề nên rất đáng tin để làm "chuẩn" kiểm tra các cách nhanh hơn (chỉ dùng với mảng nhỏ).

In [8]:
def is_possible_bruteforce(nums: List[int]) -> bool:
    n = len(nums)
    half = n // 2
    for chosen in combinations(range(n), half):          # các chỉ số của nums1
        chosen_set = set(chosen)
        nums1 = [nums[i] for i in chosen]
        nums2 = [nums[i] for i in range(n) if i not in chosen_set]
        if len(set(nums1)) == half and len(set(nums2)) == half:
            return True
    return False


for nums, expected in examples:
    print(is_possible_bruteforce(nums), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp (Cách 1).** Có `C(n, n/2)` cách chia, mỗi cách tốn `O(n)`. Với `n = 100`, `C(100, 50)` khoảng `10^29`: **không thể chạy**. Chỉ dùng được khi `n <= 20` hoặc so. **Space `O(n)`**.

**Nút thắt:** ta thử cả những cách chia rõ ràng sai (đưa hai số `1` vào cùng một nhóm). Nếu hiểu được **điều kiện nào quyết định** thì không cần thử gì cả.

## 6. Ý tưởng thuật toán / bước đột phá

**Định lý.** Có thể chia mảng (độ dài chẵn) ⇔ **mỗi giá trị xuất hiện tối đa 2 lần**.

**Chiều "cần" (⇒).** Nếu một giá trị xuất hiện `>= 3` lần: có hai nhóm nhưng ba bản sao ⇒ một nhóm chứa hai bản sao (chuồng bồ câu) ⇒ trùng ⇒ không chia được.

**Chiều "đủ" (⇐).** Giả sử mọi giá trị xuất hiện `1` hoặc `2` lần. Gọi `d2` là số giá trị xuất hiện `2` lần, `d1` là số giá trị xuất hiện `1` lần. Khi đó `n = 2·d2 + d1`. Vì `n` chẵn nên `d1` **chẵn**. Xây dựng:

- Mỗi giá trị xuất hiện 2 lần: đặt **một bản vào `nums1`, một bản vào `nums2`** (không trùng trong từng nhóm).
- Các giá trị xuất hiện 1 lần: có `d1` (số chẵn) giá trị; đặt **một nửa (`d1/2`) vào `nums1`, một nửa vào `nums2`**.

Mỗi nhóm có `d2 + d1/2 = n/2` phần tử, và không nhóm nào có hai phần tử bằng nhau. Vậy chia được.

Nhờ định lý, bài toán thành: **kiểm tra `max(số lần xuất hiện) <= 2`**.

In [9]:
# Kiểm chứng bằng tay phép xây dựng ở chiều "đủ" trên một ví dụ
nums = [5, 5, 7, 7, 8, 9, 10, 11]
cnt = Counter(nums)
twos = [v for v, c in cnt.items() if c == 2]
ones = [v for v, c in cnt.items() if c == 1]
half_ones = len(ones) // 2
nums1 = twos + ones[:half_ones]
nums2 = twos + ones[half_ones:]
print("nums1 =", nums1, "| nums2 =", nums2)
assert len(nums1) == len(nums2) == len(nums) // 2
assert len(set(nums1)) == len(nums1) and len(set(nums2)) == len(nums2)
assert sorted(nums1 + nums2) == sorted(nums)
print("phép xây dựng đúng")

nums1 = [5, 7, 8, 9] | nums2 = [5, 7, 10, 11]
phép xây dựng đúng


## 7. Các cách tiếp theo

### Cách 2 — Đếm bằng `list.count` (hai vòng lặp ngầm)

**Giải thích.** Với mỗi phần tử `x`, `nums.count(x)` cho số lần xuất hiện. Nếu có `x` nào có `count(x) > 2` thì `False`. Đơn giản nhưng mỗi `count` duyệt cả mảng ⇒ `O(n²)`; với `n <= 100` vẫn rất nhanh.

In [10]:
def is_possible_count(nums: List[int]) -> bool:
    for x in nums:
        if nums.count(x) > 2:          # x xuất hiện từ 3 lần trở lên
            return False
    return True


for nums, expected in examples:
    print(is_possible_count(nums), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


In [11]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionCount:
    def isPossibleToSplit(self, nums: List[int]) -> bool:
        return all(nums.count(x) <= 2 for x in nums)


print(SolutionCount().isPossibleToSplit([1, 1, 1, 1]))     # False

False


**Độ phức tạp Cách 2:** **Time `O(n²)`**, **Space `O(1)`**.

### Cách 3 — `Counter` (một lượt đếm)

**Giải thích.** `Counter(nums)` đếm mọi giá trị trong `O(n)`. Điều kiện: giá trị đếm lớn nhất `<= 2`. Vì `nums` không rỗng (`n >= 2`), `max` an toàn.

In [12]:
def is_possible_counter(nums: List[int]) -> bool:
    cnt = Counter(nums)
    return max(cnt.values()) <= 2


for nums, expected in examples:
    print(is_possible_counter(nums), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


In [13]:
# Code nộp LeetCode cho Cách 3
class SolutionCounter:
    def isPossibleToSplit(self, nums: List[int]) -> bool:
        return max(Counter(nums).values()) <= 2


print(SolutionCounter().isPossibleToSplit([1, 1, 2, 2, 3, 4]))     # True

True


**Độ phức tạp Cách 3:** **Time `O(n)`** (một lượt đếm), **Space `O(k)`** với `k` là số giá trị khác nhau (`k <= n`).

### Cách 4 — Sắp xếp rồi so `s[i]` với `s[i + 2]`

**Giải thích.** Sau `sorted`, các giá trị bằng nhau đứng liền nhau. Một giá trị xuất hiện từ 3 lần thì có `i` với `s[i] == s[i+1] == s[i+2]`, tức `s[i] == s[i + 2]` (vì mảng đã tăng dần). Ngược lại nếu `s[i] == s[i+2]` thì ba số liên tiếp bằng nhau. Không cần dict.

In [14]:
def is_possible_sorted(nums: List[int]) -> bool:
    s = sorted(nums)
    for i in range(len(s) - 2):
        if s[i] == s[i + 2]:               # ba số liên tiếp bằng nhau
            return False
    return True


for nums, expected in examples:
    print(is_possible_sorted(nums), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp Cách 4:** **Time `O(n log n)`** (sắp xếp), **Space `O(n)`** (bản sao đã sắp xếp; nếu sắp xếp tại chỗ thì `O(1)` phụ).

### Cách 5 — Mô phỏng việc xếp vào hai tập (`seen1`, `seen2`)

**Giải thích.** Duyệt từng số `x`: nếu `x` chưa có trong `nums1` thì cho vào `nums1`; nếu có rồi mà chưa có trong `nums2` thì cho vào `nums2`; nếu cả hai đều có rồi thì **không còn chỗ** ⇒ `False`. Cách này không cần định lý về cân bằng kích thước: ta chỉ kiểm tra rằng không có giá trị nào cần ba chỗ. (Cân bằng kích thước đã được đảm bảo ở §6.)

In [15]:
def is_possible_sets(nums: List[int]) -> bool:
    seen1, seen2 = set(), set()
    for x in nums:
        if x not in seen1:
            seen1.add(x)
        elif x not in seen2:
            seen2.add(x)
        else:
            return False               # x đã có ở cả hai nhóm
    return True


for nums, expected in examples:
    print(is_possible_sets(nums), "(kỳ vọng", expected, ")")

True (kỳ vọng True )
False (kỳ vọng False )
True (kỳ vọng True )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(k)`**.

## 8. TRACE từng bước

Ta trace Cách 5 (hai tập) và Cách 4 (sắp xếp) trên `nums = [1, 1, 2, 2, 3, 4]` rồi trên `nums = [1, 2, 1, 3, 1, 4]` (có ba số `1`).

In [16]:
def trace_sets(nums: List[int]) -> bool:
    seen1, seen2 = set(), set()
    print("nums =", nums)
    for i, x in enumerate(nums):
        if x not in seen1:
            action = "vào nhóm 1"
            seen1.add(x)
        elif x not in seen2:
            action = "vào nhóm 2"
            seen2.add(x)
        else:
            print(f"i={i} x={x}: cả hai nhóm đều có {x} -> False")
            return False
        print(f"i={i} x={x}: {action} | nhóm1={sorted(seen1)} nhóm2={sorted(seen2)}")
    print("không số nào cần chỗ thứ 3 -> True")
    return True


assert trace_sets([1, 1, 2, 2, 3, 4]) is True
print()
assert trace_sets([1, 2, 1, 3, 1, 4]) is False

nums = [1, 1, 2, 2, 3, 4]
i=0 x=1: vào nhóm 1 | nhóm1=[1] nhóm2=[]
i=1 x=1: vào nhóm 2 | nhóm1=[1] nhóm2=[1]
i=2 x=2: vào nhóm 1 | nhóm1=[1, 2] nhóm2=[1]
i=3 x=2: vào nhóm 2 | nhóm1=[1, 2] nhóm2=[1, 2]
i=4 x=3: vào nhóm 1 | nhóm1=[1, 2, 3] nhóm2=[1, 2]
i=5 x=4: vào nhóm 1 | nhóm1=[1, 2, 3, 4] nhóm2=[1, 2]
không số nào cần chỗ thứ 3 -> True

nums = [1, 2, 1, 3, 1, 4]
i=0 x=1: vào nhóm 1 | nhóm1=[1] nhóm2=[]
i=1 x=2: vào nhóm 1 | nhóm1=[1, 2] nhóm2=[]
i=2 x=1: vào nhóm 2 | nhóm1=[1, 2] nhóm2=[1]
i=3 x=3: vào nhóm 1 | nhóm1=[1, 2, 3] nhóm2=[1]
i=4 x=1: cả hai nhóm đều có 1 -> False


In [17]:
def trace_sorted(nums: List[int]) -> bool:
    s = sorted(nums)
    print("sau sort:", s)
    for i in range(len(s) - 2):
        eq = s[i] == s[i + 2]
        print(f"i={i}: s[{i}]={s[i]}, s[{i + 2}]={s[i + 2]} -> {'bằng nhau => False' if eq else 'khác'}")
        if eq:
            return False
    return True


assert trace_sorted([1, 1, 2, 2, 3, 4]) is True
print()
assert trace_sorted([1, 2, 1, 3, 1, 4]) is False

sau sort: [1, 1, 2, 2, 3, 4]
i=0: s[0]=1, s[2]=2 -> khác
i=1: s[1]=1, s[3]=2 -> khác
i=2: s[2]=2, s[4]=3 -> khác
i=3: s[3]=2, s[5]=4 -> khác

sau sort: [1, 1, 1, 2, 3, 4]
i=0: s[0]=1, s[2]=1 -> bằng nhau => False


Bảng trace Cách 5 với `[1, 1, 2, 2, 3, 4]`:

| i | x | seen1 trước | seen2 trước | hành động |
|---|---|---|---|---|
| 0 | 1 | {} | {} | vào nhóm 1 |
| 1 | 1 | {1} | {} | đã có ở nhóm 1, vào nhóm 2 |
| 2 | 2 | {1} | {1} | vào nhóm 1 |
| 3 | 2 | {1,2} | {1} | vào nhóm 2 |
| 4 | 3 | {1,2} | {1,2} | vào nhóm 1 |
| 5 | 4 | {1,2,3} | {1,2} | vào nhóm 1 (nhóm này chỉ là "tập đã thấy", không phải chia cuối cùng) |

Hai tập `seen1` và `seen2` chỉ là công cụ ghi nhớ "giá trị này đã dùng bao nhiêu lần (0, 1, hay 2)". Chia thực sự (cân bằng kích thước) được xây ở §6.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (`Counter`): ngắn, rõ, `O(n)`.

In [18]:
from typing import List
from collections import Counter

class Solution:
    def isPossibleToSplit(self, nums: List[int]) -> bool:
        # mỗi giá trị xuất hiện tối đa 2 lần <=> chia được hai nửa không trùng
        return max(Counter(nums).values()) <= 2


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.isPossibleToSplit(nums))

[1, 1, 2, 2, 3, 4] -> True
[1, 1, 1, 1] -> False
[1, 2] -> True


## 10. Vì sao thuật toán đúng

- **Cần:** nếu có giá trị xuất hiện `>= 3` lần thì hai nhóm không đủ chứa chúng mà không trùng (chuồng bồ câu). Khi đó đáp án chắc chắn `False`, và thuật toán trả `False`.
- **Đủ:** nếu mọi giá trị xuất hiện `<= 2` lần, ta có cách xây dựng cụ thể ở §6 cho hai nhóm cùng kích thước `n/2`, không trùng. Khi đó đáp án `True`, và thuật toán trả `True`.
- **Vai trò của "độ dài chẵn":** làm cho số giá trị xuất hiện 1 lần (`d1`) chẵn, nên chia đều được. Nếu `n` lẻ thì đề bài không đặt ra (hai nửa bằng nhau không tồn tại).
- Hai tiêu chí cho cùng một kết quả: Cách 1 (định nghĩa) khớp các cách 2-5 trên mọi input nhỏ (xem §12.3).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `k` là số giá trị phân biệt.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cách chia | `O(C(n, n/2) · n)` | `O(n)` |
| 2. `list.count` | `O(n²)` | `O(1)` |
| 3. `Counter` | `O(n)` | `O(k)` |
| 4. Sắp xếp, so `s[i]` với `s[i+2]` | `O(n log n)` | `O(n)` |
| 5. Hai tập `seen1`, `seen2` | `O(n)` | `O(k)` |

Với `n <= 100` mọi cách trừ Cách 1 đều dưới một mili-giây.

## 12. Bộ test

Ta gom các cách nhanh vào `impls`, còn Cách 1 chỉ dùng cho input nhỏ.

In [19]:
sol = Solution()
impls = {
    "count": is_possible_count,
    "counter": is_possible_counter,
    "sorted": is_possible_sorted,
    "sets": is_possible_sets,
    "leetcode": sol.isPossibleToSplit,
}


def check_all(nums, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums))
        assert actual == expected, (label, key, nums, actual, expected)
    if len(nums) <= 14:
        assert is_possible_bruteforce(list(nums)) == expected, (label, "brute", nums)
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [20]:
check_all([1, 2], True, "n = 2, hai số khác nhau")
check_all([5, 5], True, "n = 2, hai số bằng nhau (mỗi nhóm một số 5)")
check_all([1, 1, 1, 2], False, "ba số 1")
check_all([1, 1, 2, 2], True, "hai cặp")
check_all([7, 7, 7, 7, 7, 7], False, "toàn số giống nhau, n = 6")
check_all([1, 2, 3, 4, 5, 6], True, "toàn số khác nhau")
check_all(list(range(1, 51)) * 2, True, "n = 100: 50 giá trị, mỗi giá trị đúng 2 lần")
check_all(list(range(1, 51)) * 2 + [], True, "như trên, lặp lại để chắc chắn")
check_all([100] * 3 + list(range(1, 98)), False, "n = 100 với ba số 100")
check_all(list(range(1, 101)), True, "n = 100, 100 giá trị khác nhau")

PASS n = 2, hai số khác nhau
PASS n = 2, hai số bằng nhau (mỗi nhóm một số 5)
PASS ba số 1
PASS hai cặp
PASS toàn số giống nhau, n = 6
PASS toàn số khác nhau
PASS n = 100: 50 giá trị, mỗi giá trị đúng 2 lần
PASS như trên, lặp lại để chắc chắn
PASS n = 100 với ba số 100
PASS n = 100, 100 giá trị khác nhau


### 12.2. Phản ví dụ cho các cách làm sai

Hai "ý tưởng gần đúng": (a) **chỉ đếm số giá trị khác nhau**: `len(set(nums)) >= n / 2`; (b) **chia theo vị trí**: nửa đầu và nửa sau đều không trùng. Ta tìm input làm lộ lỗi.

In [21]:
def wrong_distinct_count(nums):
    return len(set(nums)) >= len(nums) // 2         # chỉ xét số giá trị khác nhau


def wrong_first_second_half(nums):
    h = len(nums) // 2
    a, b = nums[:h], nums[h:]
    return len(set(a)) == len(a) and len(set(b)) == len(b)


bad1 = [1, 1, 1, 2, 3, 4]       # 4 giá trị khác nhau >= 3, nhưng có ba số 1
bad2 = [1, 1, 2, 2, 3, 4]       # chia nửa đầu/nửa sau: [1,1,2] có trùng, nhưng vẫn chia được
print(wrong_distinct_count(bad1), "!=", is_possible_counter(bad1))
print(wrong_first_second_half(bad2), "!=", is_possible_counter(bad2))
assert wrong_distinct_count(bad1) != is_possible_counter(bad1)
assert wrong_first_second_half(bad2) != is_possible_counter(bad2)

True != False
False != True


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Với mảng nhỏ ta so mọi cách nhanh với Cách 1 (định nghĩa). Với mảng lớn đúng ràng buộc đề, ta so các cách nhanh với nhau.

In [22]:
rng = random.Random(3046)
for trial in range(1500):
    n = rng.choice([2, 4, 6, 8, 10, 12])
    hi = rng.choice([2, 3, 4, 6, 12])
    nums = [rng.randint(1, hi) for _ in range(n)]
    expected = is_possible_bruteforce(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == expected, (key, nums)
print("PASS 1500 mảng nhỏ khớp brute force theo định nghĩa")

for trial in range(1000):
    n = rng.choice([2, 10, 50, 98, 100])
    hi = rng.choice([2, 10, 50, 100])
    nums = [rng.randint(1, hi) for _ in range(n)]
    assert all(1 <= x <= 100 for x in nums) and 1 <= len(nums) <= 100 and len(nums) % 2 == 0
    ref = is_possible_counter(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
print("PASS 1000 mảng lớn đúng ràng buộc: các cách nhanh khớp nhau")

PASS 1500 mảng nhỏ khớp brute force theo định nghĩa
PASS 1000 mảng lớn đúng ràng buộc: các cách nhanh khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ so số giá trị khác nhau với `n / 2`

In [23]:
def sai_dem_gia_tri_khac_nhau(nums):
    return len(set(nums)) >= len(nums) // 2


nums = [1, 1, 1, 2, 3, 4]
print("sai:", sai_dem_gia_tri_khac_nhau(nums), "| đúng:", Solution().isPossibleToSplit(nums))

sai: True | đúng: False


**Vì sao sai:** đủ "nhiều giá trị khác nhau" chưa đảm bảo, vì một giá trị có thể lặp nhiều lần (ba số `1` ở trên). Điều kiện quyết định là **số lần lặp tối đa**, không phải số giá trị khác nhau. **Cách sửa:** `max(Counter(nums).values()) <= 2`.

### Lỗi 2: Nhầm "tối đa 2 lần" thành "đúng 2 lần"

In [24]:
def sai_dung_hai_lan(nums):
    return all(c == 2 for c in Counter(nums).values())      # SAI: [1, 2] bị loại


print("sai:", sai_dung_hai_lan([1, 2]), "| đúng:", Solution().isPossibleToSplit([1, 2]))

sai: False | đúng: True


**Vì sao sai:** giá trị xuất hiện **một** lần hoàn toàn hợp lệ (nó chỉ nằm ở một nhóm). Điều kiện là `<= 2`, không phải `== 2`. **Cách sửa:** so sánh `<= 2`.

### Lỗi 3: Chia theo vị trí (nửa đầu / nửa sau)

In [25]:
def sai_chia_theo_vi_tri(nums):
    h = len(nums) // 2
    return len(set(nums[:h])) == h and len(set(nums[h:])) == h


nums = [1, 1, 2, 2, 3, 4]
print("sai:", sai_chia_theo_vi_tri(nums), "| đúng:", Solution().isPossibleToSplit(nums))

sai: False | đúng: True


**Vì sao sai:** đề cho phép **chia tự do** (mỗi phần tử vào nhóm nào cũng được), không bắt buộc nửa trái / nửa phải. Với `[1, 1, 2, 2, 3, 4]` nửa đầu `[1, 1, 2]` trùng nhưng vẫn có cách chia khác. **Cách sửa:** đưa về điều kiện số lần xuất hiện.

### Lỗi 4: Dùng `list.count` rồi quên rằng đó là `O(n)` mỗi lần (và `if` sai hướng)

In [26]:
def sai_huong_dieu_kien(nums):
    for x in nums:
        if nums.count(x) < 2:          # SAI: trả False khi phần tử chỉ xuất hiện 1 lần
            return False
    return True


print("sai:", sai_huong_dieu_kien([1, 2]), "| đúng:", Solution().isPossibleToSplit([1, 2]))

sai: False | đúng: True


**Vì sao sai:** điều kiện bị đảo: ta phải loại khi `count > 2`, chứ không phải khi `count < 2`. Mẹo kiểm tra: thử `[1, 2]` (toàn giá trị xuất hiện một lần) — kết quả đúng là `True`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chia vào **K nhóm, mỗi nhóm không trùng** ⇒ mỗi giá trị xuất hiện **tối đa K lần**. Ở đây `K = 2`. Bài tổng quát: chia mảng thành `K` tập con không trùng ⇒ `max(count) <= K` (còn phải kiểm kích thước).
- Thấy "xuất hiện bao nhiêu lần" ⇒ nghĩ `Counter` ngay.
- Thấy ràng buộc **chẵn** ⇒ thường có lý do cân bằng ẩn (ở đây làm `d1` chẵn).
- Nếu bạn muốn tự tin: viết **brute force theo định nghĩa** cho input nhỏ rồi so với công thức; đây là cách tìm ra định lý này mà không cần chứng minh trước.
- Chuồng bồ câu: `m` bồ câu, `k < m` chuồng ⇒ có chuồng `>= 2` con.

## 15. Biến thể của bài

1. **Chia thành 3 nhóm** (độ dài chia hết cho 3): điều kiện thành `max(count) <= 3` và vẫn cần cân bằng.
2. **Trả về một cách chia cụ thể** `(nums1, nums2)` (không chỉ `True/False`).
3. **Không bắt buộc hai nhóm bằng nhau** về kích thước: điều kiện cũng là `max(count) <= 2`, và còn dễ hơn.
4. **Số cách chia khác nhau** (đếm cách): bài tổ hợp, cần công thức.
5. **Contains Duplicate (217)** và **Find All Duplicates in an Array (442)**: cùng chủ đề đếm trùng.

In [27]:
def split_array(nums):
    """Trả về (nums1, nums2) hợp lệ hoặc None nếu không chia được."""
    cnt = Counter(nums)
    if max(cnt.values()) > 2:
        return None
    twos = [v for v, c in cnt.items() if c == 2]
    ones = [v for v, c in cnt.items() if c == 1]
    h = len(ones) // 2
    return twos + ones[:h], twos + ones[h:]


r = split_array([1, 1, 2, 2, 3, 4])
print(r)
assert r is not None and len(r[0]) == len(r[1]) == 3
assert all(len(set(g)) == len(g) for g in r)
assert split_array([1, 1, 1, 1]) is None

rng = random.Random(46)
for _ in range(500):
    nums = [rng.randint(1, 8) for _ in range(rng.choice([2, 4, 6, 8, 12]))]
    r = split_array(nums)
    assert (r is not None) == is_possible_counter(nums)
    if r is not None:
        assert sorted(r[0] + r[1]) == sorted(nums) and len(r[0]) == len(r[1])
        assert all(len(set(g)) == len(g) for g in r)
print("PASS biến thể: dựng cách chia")

([1, 2, 3], [1, 2, 4])
PASS biến thể: dựng cách chia


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cách chia | bám định nghĩa | `O(C(n, n/2) · n)` | `O(n)` | chỉ làm "chuẩn" kiểm tra cho input nhỏ |
| 2. `list.count` | mỗi phần tử đếm lại | `O(n²)` | `O(1)` | `n` rất nhỏ, ít code nhất |
| 3. `Counter` | đếm một lượt | `O(n)` | `O(k)` | **nên nộp**, rõ nhất |
| 4. Sắp xếp | `s[i] == s[i+2]` | `O(n log n)` | `O(n)` | không muốn dùng dict |
| 5. Hai tập | `seen1`, `seen2` | `O(n)` | `O(k)` | muốn mô phỏng việc xếp nhóm |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Counter([1, 1, 2])`, `[1, 2, 2].count(2)`, `len(set([4, 4, 4]))`, `all(c <= 2 for c in [1, 2, 3])`, `Solution().isPossibleToSplit([3, 3, 3, 3, 1, 2])`.

**Bài 2 (tính tay).** Với `nums = [4, 4, 5, 6, 6, 7, 8, 9]`: tìm `d2` (số giá trị xuất hiện 2 lần), `d1` (số giá trị xuất hiện 1 lần), xác nhận `n = 2·d2 + d1`, và viết ra một cách chia hợp lệ.

**Bài 3 (điền TODO).** Hoàn thành `is_possible_todo(nums)` theo Cách 5 (hai tập).

In [28]:
def is_possible_todo(nums):
    seen1, seen2 = set(), set()
    for x in nums:
        # TODO 1: nếu x chưa có trong seen1 -> thêm vào seen1
        # TODO 2: ngược lại, nếu x chưa có trong seen2 -> thêm vào seen2
        # TODO 3: ngược lại (cả hai đều có) -> return False
        pass
    return True

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `is_possible_sorted2(nums)` theo Cách 4 (sắp xếp, so `s[i]` với `s[i + 2]`) **mà không** dùng `Counter`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `is_possible_k(nums, k)` kiểm tra có thể chia `nums` thành **`k` nhóm** (`len(nums) % k == 0`, mỗi nhóm cỡ `len(nums) // k`, không trùng trong nhóm) — điều kiện là `max(count) <= k`. (b) Viết `min_groups(nums)` trả về **số nhóm ít nhất** (không cần cân bằng kích thước) để mỗi nhóm không trùng; đáp án chính là `max(count)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `Counter({1: 2, 2: 1})`, `2`, `1`, `False` (vì `3 <= 2` sai), `False`. Bài 3: ba nhánh `if / elif / else`; nhánh `else` trả `False`.
- **Mức 2 (vừa):** Bài 2: `4` xuất hiện 2 lần, `6` xuất hiện 2 lần ⇒ `d2 = 2`; các giá trị `5, 7, 8, 9` xuất hiện 1 lần ⇒ `d1 = 4`; `2·2 + 4 = 8`. Chia: `[4, 6, 5, 7]` và `[4, 6, 8, 9]`. Bài 4: `for i in range(len(s) - 2)`.
- **Mức 3 (sâu):** Bài 5(a): thêm điều kiện chia hết rồi so `max(Counter(nums).values()) <= k`. Với `k` nhóm bằng nhau mà các giá trị xuất hiện 1 lần nhiều thì có thể rải đều được vì tổng chia hết cho `k` (kiểm tra bằng brute force nhỏ nếu muốn chắc). 5(b): mỗi giá trị xuất hiện `c` lần cần `c` nhóm khác nhau, nên cần ít nhất `max(count)` nhóm, và `max(count)` nhóm là đủ (đặt bản sao thứ `t` vào nhóm `t`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert Counter([1, 1, 2]) == Counter({1: 2, 2: 1})
assert [1, 2, 2].count(2) == 2
assert len(set([4, 4, 4])) == 1
assert all(c <= 2 for c in [1, 2, 3]) is False
assert Solution().isPossibleToSplit([3, 3, 3, 3, 1, 2]) is False

# Bài 2
nums2 = [4, 4, 5, 6, 6, 7, 8, 9]
cnt2 = Counter(nums2)
d2 = sum(1 for c in cnt2.values() if c == 2)
d1 = sum(1 for c in cnt2.values() if c == 1)
assert (d2, d1) == (2, 4) and 2 * d2 + d1 == len(nums2)
g1, g2 = [4, 6, 5, 7], [4, 6, 8, 9]
assert sorted(g1 + g2) == sorted(nums2) and len(set(g1)) == 4 and len(set(g2)) == 4
assert Solution().isPossibleToSplit(nums2) is True


# Bài 3
def is_possible_todo_ans(nums):
    seen1, seen2 = set(), set()
    for x in nums:
        if x not in seen1:
            seen1.add(x)
        elif x not in seen2:
            seen2.add(x)
        else:
            return False
    return True


for nums, expected in examples:
    assert is_possible_todo_ans(nums) == expected


# Bài 4
def is_possible_sorted2(nums):
    s = sorted(nums)
    for i in range(len(s) - 2):
        if s[i] == s[i + 2]:
            return False
    return True


for nums, expected in examples:
    assert is_possible_sorted2(nums) == expected


# Bài 5(a)
def is_possible_k(nums, k):
    if len(nums) % k != 0:
        return False
    return max(Counter(nums).values()) <= k


assert is_possible_k([1, 1, 2, 2, 3, 4], 2) is True
assert is_possible_k([1, 1, 1, 2, 2, 2], 3) is True
assert is_possible_k([1, 1, 1, 1, 2, 3], 3) is False
assert is_possible_k([1, 2, 3], 2) is False      # 3 không chia hết cho 2


# Bài 5(b)
def min_groups(nums):
    return max(Counter(nums).values())


assert min_groups([1, 1, 1, 2]) == 3
assert min_groups([1, 2, 3]) == 1
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chia thành `K` nhóm không trùng ⇒ giá trị xuất hiện tối đa `K` lần; ở đây `K = 2`)

```python
from typing import List
from collections import Counter

class Solution:
    def isPossibleToSplit(self, nums: List[int]) -> bool:
        return max(Counter(nums).values()) <= 2
```

**Checklist**

- [ ] Điều kiện là số lần xuất hiện **`<= 2`** (không phải `== 2`, không phải đếm giá trị khác nhau)?
- [ ] Đã hiểu vì sao **đủ** (xây cách chia) và vì sao **cần** (chuồng bồ câu)?
- [ ] Không chia theo vị trí nửa đầu/nửa sau?
- [ ] Đã thử `[1, 1, 1, 1]`, `[1, 2]`, `[5, 5]`?
- [ ] Dùng `Counter` (một lượt đếm) thay vì `list.count` trong vòng lặp nếu `n` lớn?
- [ ] Skeleton đúng `class Solution` / `def isPossibleToSplit(self, nums: List[int]) -> bool`?